# OmniEvo - ejemplo basico

como usar el paquete para optimizar atribucion con GA

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from omnievo import (
    DataGenerator,
    GeneticOptimizer,
    compare_baselines,
    plot_convergence,
    plot_weights,
    plot_comparison,
)
from omnievo.fitness import predict_ltv

## generar datos

simulamos un festival con usuarios de diferentes segmentos

In [ ]:
gen = DataGenerator(n_users=1000, random_state=42)
df = gen.generate()
channels = gen.get_channel_names()

print(f"{len(df)} usuarios, {len(channels)} canales")
print(df['segment'].value_counts())
df.head()

In [ ]:
X = df[channels].values
y = df['LTV_real'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"train: {len(X_train)}, test: {len(X_test)}")

## correr el GA

In [ ]:
opt = GeneticOptimizer(
    population_size=50,
    generations=50,
    random_state=42,
)

result = opt.fit(X_train, y_train)
print(f"fitness: {result['best_fitness']:.4f}")

In [ ]:
# pesos
print("Pesos:")
for ch, w in sorted(zip(channels, result['best_weights']), key=lambda x: -x[1]):
    print(f"  {ch}: {w:.3f} ({w*100:.1f}%)")

## graficas

In [ ]:
plot_convergence(result);

In [ ]:
plot_weights(result['best_weights'], channels);

## comparar con baselines

In [ ]:
comp = compare_baselines(X_test, y_test, ga_weights=result['best_weights'])
comp

In [ ]:
plot_comparison(comp);

## conclusion

el GA encuentra pesos que predicen mejor el LTV que los modelos tipicos (uniforme, last-touch, etc). los canales IoT suelen tener mas peso porque capturan comportamiento real de consumo.